In [ ]:
from dataclasses import dataclass
import numpy as np

# Trajectory Container Tools - Direct Instanciation Usage Example

This notebook demonstrates the essential usage of Trajectory Container Tools (TCT) dataclasses for creating and working with trajectory data.

## Import trajectory-container-tools namespace


In [ ]:
import trajectory_container_tools as tct

## 1. Basic Trajectory Creation

### Simple Trajectory Container Structure

The most straightforward way to create a trajectory is by extending `BaseTrajectoryDataclass`:

In [ ]:
# Generate sample trajectory data
timesteps = 50
time_array = np.linspace(0, 5, timesteps)
x_positions = np.sin(time_array)
y_positions = np.cos(time_array)
timestamps = np.arange(timesteps) * 0.1  # 10 Hz sampling

@dataclass()
class Simple2DCoordinateTrajectory(tct.BaseTrajectoryDataclass):
    x: np.ndarray
    y: np.ndarray
    timestamps: np.ndarray

# Create trajectory container
trajectory = Simple2DCoordinateTrajectory(
    feature_name="2D coordinate",
    x=x_positions,
    y=y_positions,
    timestamps=timestamps
)

print((
        f"Created trajectory with {trajectory.trajectory_len} timesteps\n"
        f"Available dimensions: {trajectory.get_dimension_names()}\n"
), trajectory)


### Nested Trajectory Container Structure

For more complex data organization, use `NestedBaseTrajectoryDataclass` for custom implementation or use dataclasses from `primitive_dataclass` module:

In [ ]:
from trajectory_container_tools.dataclasses import Vector2D

@dataclass()
class CustomPoseContainer(tct.NestedBaseTrajectoryDataclass):
    x: np.ndarray
    y: np.ndarray


@dataclass()
class ComplexTrajectory(tct.BaseTrajectoryDataclass):
    timestamps: np.ndarray
    position: CustomPoseContainer
    velocity: Vector2D


sin_cos_trajectory_object = ComplexTrajectory(feature_name="mock",
                                              timestamps=timestamps,
                                              position=CustomPoseContainer(x_positions, y_positions),
                                              velocity=Vector2D(np.ones_like(x_positions), np.ones_like(y_positions))
                                              )

print(sin_cos_trajectory_object)

## 2. Factory-Based Creation

TCT provides factory functions for dynamic trajectory dataclass creation:

In [ ]:
mock_data = np.random.randn(100, 4)  # 100 timesteps, 4 dimensions

# Define the specification
spec = tct.factory.TrjDataClassFeatureSpecification(
        new_feature_dataclass_type='DynamicTrajectory',
        dimension_names=('x', 'y', 'velocity', 'acceleration')
        )

# Create the dataclass type
DynamicTrajectory = tct.factory.create_dataclass(specification=spec)

# Use the dynamically created class
factory_generated_trajectory = DynamicTrajectory(
        feature_name="Factory-made-mock-trajectory",
        x=mock_data[:, 0],
        y=mock_data[:, 1],
        velocity=mock_data[:, 2],
        acceleration=mock_data[:, 3]
        )

print(factory_generated_trajectory)

## 3. Accessing Trajectory Data

Demonstrate basic data access and manipulation.


In [ ]:
# Access individual dimensions
print(f"X position range: [{ trajectory.x.min():.2f}, { trajectory.x.max():.2f}]")
print(f"Y position range: [{ trajectory.y.min():.2f}, { trajectory.y.max():.2f}]")
print(f"Time range: [{ trajectory.timestamps.min():.2f}, { trajectory.timestamps.max():.2f}] seconds")

## 4. Trajectory Slicing

Extract portions of the trajectory.


In [ ]:
# Slice trajectory (get timesteps 10-30)
partial_trajectory = trajectory[10:30]
print(f"Original trajectory length: {trajectory.trajectory_len}")
print(f"Partial trajectory length: {partial_trajectory.trajectory_len}")

# Access sliced data
print(f"Partial X range: [{partial_trajectory.x.min():.2f}, {partial_trajectory.x.max():.2f}]")

## 5. Iterating Through Trajectory Points

Iterate through trajectory data points.


In [ ]:
# Iterate through first 5 trajectory points
print("First 5 trajectory points:")
for i, point in enumerate(trajectory):
    if i >= 5:
        break
    print(f"Point {i}: x={point.x:.3f}, y={point.y:.3f}")


## 6. Trajectory Batching

In [ ]:
# Create batch trajectories (3 trajectories, 20 timesteps each)
batch_size, time_steps = 3, 20
batch_x = np.random.randn(batch_size, time_steps)
batch_y = np.random.randn(batch_size, time_steps)
batch_frame = np.random.randn(batch_size, time_steps, 10)
batch_timestamps = np.tile(np.arange(time_steps) * 0.1, (batch_size, 1))

@dataclass()
class Simple2DCoordinateTrajectory(tct.BaseTrajectoryDataclass):
    x: np.ndarray
    y: np.ndarray
    frame: np.ndarray
    timestamps: np.ndarray


batch_trajectory = Simple2DCoordinateTrajectory(
    feature_name="batch 2d coordinate",
    x=batch_x,
    y=batch_y,
    frame=batch_frame,
    timestamps=batch_timestamps,
    batch=True
)

print(f"Batch trajectory shape: {batch_trajectory.x.shape}")
print(f"Number of trajectories: {batch_trajectory.x.shape[0]}")
print(f"Timesteps per trajectory: {batch_trajectory.x.shape[1]}")
print(f"Trajectory length: {len(batch_trajectory)}")

print(batch_trajectory)

## 7. Data Validation

The dataclasses automatically validate data consistency.


In [ ]:
# Example of data validation - this will work
trajectory_length = 5
valid_x = np.arange(trajectory_length)
valid_y = np.arange(trajectory_length)
valid_frame = np.random.randn(trajectory_length, 10)
valid_timestamps = np.arange(trajectory_length) * 0.1


# Create trajectory container
valid_trajectory = Simple2DCoordinateTrajectory(
    feature_name="2D coordinate – valid",
    x=valid_x,
    y=valid_y,
    frame=valid_frame,
    timestamps=valid_timestamps
)

print("Valid trajectory created successfully!")
print(f"Trajectory length: {valid_trajectory.trajectory_len}\n")

# Demonstrate error handling with mismatched dimensions
try:
    # This should raise an error due to mismatched array lengths
    invalid_y = np.arange(trajectory_length - 1)  # 4 elements - mismatch!

    invalid_trajectory = Simple2DCoordinateTrajectory(
        feature_name="2D coordinate – invalid",
        x=valid_x,
        y=invalid_y,
        frame=valid_frame,
        timestamps=valid_timestamps
    )

except ValueError as e:
    print(f"Expected error caught: {type(e).__name__}")
    print(e)

## Summary

This notebook covered the essential usage patterns of TCT dataclasses:

1. **Basic trajectory creation**
2. **Factory-based creation**
3. **Data access** and manipulation methods
4. **Trajectory slicing** for extracting portions of data
5. **Iteration** through trajectory points
6. **Trajectory batching**
7. **Data validation** and error handling

For more examples, refer to the documentation at `documentation/direct_instantiation.md`.
